# Product Scraping Agent — single URL run

This notebook uses the latest input contract:

- `product_url` is the actual URL to scrape.
- `main_text` and `ean` are product identity hints.
- requested retailer/country are the original business target.
- source retailer/country describe the actual URL source, which may be a fallback/global retailer.

The scraper does **not** search. It builds a product evidence artifact from the supplied URL and supplied product context.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_PATH = PROJECT_ROOT / 'src'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_PATH:', SRC_PATH)
print('SRC exists:', SRC_PATH.exists())


In [ ]:
from dotenv import load_dotenv

env_path = PROJECT_ROOT / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print('Loaded .env:', env_path)
else:
    print('No .env found. Copy .env.example to .env and fill LLM/browser settings if needed.')


In [ ]:
# Optional: clear already-loaded package modules after replacing code in the same kernel.
# Use this cell only if you updated the repository without restarting the notebook kernel.
import sys
for name in list(sys.modules):
    if name.startswith('product_scraping_agent'):
        del sys.modules[name]
print('Cleared product_scraping_agent modules from kernel cache')


In [ ]:
from product_scraping_agent import ProductScrapingAgent, ScrapeRequest
from product_scraping_agent.models import ScrapeResult
import inspect

print('Import successful')
print('ScrapeResult file:', inspect.getfile(ScrapeResult))
print('Has artifact_quality:', 'artifact_quality' in ScrapeResult.model_fields)
print('Has image_candidate_count:', 'image_candidate_count' in ScrapeResult.model_fields)


## Define request

Use the URL you actually want to scrape. If the URL is a fallback source, keep `source_url_role` as `global_fallback`, `marketplace_fallback`, or another appropriate role. This does not stop scraping; it only scopes retailer-specific claims correctly.


In [ ]:
request = ScrapeRequest(
    # Required: actual URL to scrape
    product_url='https://www.amazon.com/Barbie-Collectible-All-Denim-Signature-Underwear/dp/B0BHFD7BPM',

    # Product identity hints from your input row
    main_text='THE MOVIE 2023 KEN DOLL WEARING DENIM SET',
    ean='194735174539',

    # Original requested market context
    requested_retailer_name='Mercado Libre',
    requested_country_code='CO',

    # Actual evidence source context for the URL above
    source_retailer_name='Amazon',
    source_country_code='US',
    source_url_role='global_fallback',

    # Optional upstream/indexed evidence from the search layer.
    # Leave empty unless your discovery stage already produced it.
    upstream_ai_evidence='',
    candidate_snippets=[],
    search_evidence=[],

    output_root=PROJECT_ROOT / 'data' / 'scraped',
    max_images=30,
    vision_max=12,
    max_agent_iterations=2,
    strict_product_only=True,
    write_raw_debug=False,
)

request.model_dump()


In [ ]:
agent = ProductScrapingAgent()
result = await agent.scrape(request)
result.model_dump()


In [ ]:
print('Success:', result.success)
print('Access:', result.access_status)
print('Browser visible:', result.browser_visible)
print('Recovered:', result.product_details_recovered)
print('Recovery status:', result.recovery_status)
print('Quality:', result.artifact_quality)
print('Quality score:', result.quality_score)
print('Manual review:', result.requires_manual_review)
print('Missing fields:', result.missing_critical_fields)
print('Evidence axes:', result.evidence_axes_used)
print('Output dir:', result.output_dir)
print('
IMAGE COUNTS')
print('Image candidates discovered:', result.image_candidate_count)
print('Final clean images in images/:', result.final_image_count)
print('Image files retained:', result.image_downloaded_count)
print('Vision described:', result.vision_described_count)
print('
ARTIFACT PATHS')
print('Product evidence JSON:', result.product_evidence_json_path)
print('Product evidence MD:', result.product_evidence_md_path)
print('Claims MD:', result.claims_md_path)
print('Vision MD:', result.vision_md_path)
print('Quality report:', result.quality_report_path)
print('Source alignment:', result.source_alignment_report_path)
print('Error:', result.error)


## Inspect whether the URL truly scraped

For Amazon and similar marketplaces, HTTP 200 can still be a weak shell/soft-block page. The important indicators are:

- small `source.md`
- no tables / no JSON-LD
- zero final images
- `browser_visible=False`
- quality marked `partial` or `insufficient`

If `images/` contains files, they should be final vision-confirmed product images only. Candidate images that failed or were rejected should appear only in `manifests/image_manifest.json`.


In [ ]:
import json
from pathlib import Path

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path and path.exists() else {}

quality = read_json(result.quality_report_path)
alignment = read_json(result.source_alignment_report_path)
manifest = read_json(result.artifact_manifest_path)
image_manifest = read_json(result.image_manifest_path)

print('QUALITY REPORT')
print(json.dumps(quality, indent=2, ensure_ascii=False)[:5000])
print('
SOURCE ALIGNMENT REPORT')
print(json.dumps(alignment, indent=2, ensure_ascii=False)[:5000])
print('
ARTIFACT COUNT SUMMARY')
print(json.dumps(manifest.get('counts', {}), indent=2, ensure_ascii=False))


In [ ]:
# Inspect final clean images folder. For weak Amazon captures, this may be empty.
images_dir = Path(result.output_dir) / 'images'
files = sorted([p.name for p in images_dir.iterdir()]) if images_dir.exists() else []
print('images_dir:', images_dir)
print('Final files in images/:', files)
print('Count:', len(files))

# Show rejected/failed image candidates from manifest.
failed_or_removed = [x for x in image_manifest if x.get('error') or not x.get('local_path')]
print('
Rejected / failed image candidates:', len(failed_or_removed))
for row in failed_or_removed[:10]:
    print('-', row.get('error') or 'removed', '|', row.get('url', '')[:120])


In [ ]:
# Preview product-only source and claims.
for label, path in [
    ('SOURCE.MD', result.source_md_path),
    ('CLAIMS.MD', result.claims_md_path),
    ('VISION.MD', result.vision_md_path),
]:
    path = Path(path)
    print('
' + '=' * 100)
    print(label, path)
    print('=' * 100)
    if path.exists():
        print(path.read_text(encoding='utf-8')[:5000])
    else:
        print('missing')
